## Single-Head, Single-Layer Transformer (Simplified)

### Setup

Let $V$ denote the vocabulary size and $L$ the context length. We choose an embedding dimension

$$d = V + L$$

and define orthonormal basis vectors $e_1, \ldots, e_d \in \mathbb{R}^{d\times 1}$.

**Token embeddings** occupy the first $V$ coordinates, and **positional encodings** the remaining $L$:

$$\text{token } t \mapsto e_t, \qquad t \in \{1, \ldots, V\}$$

$$\text{position } p \mapsto e_{V+p}, \qquad p \in \{1, \ldots, L\}$$

The combined representation of token $t$ at position $p$ is

$$x(t, p) = e_t + e_{V+p} \in \mathbb{R}^{d\times 1}.$$

---

### Attention

We fix $W_Q = I\in \mathbb{R}^{d\times d}$, so the query for position $i$ is simply

$$q_i = x(t_i, p_i)\in \mathbb{R}^{d\times 1}.$$

The key and value projections are learned:

$$k_j = W_K\, x(t_j, p_j), \qquad v_j = W_V\, x(t_j, p_j), \quad \text{(both in $\mathbb{R}^{d\times 1}$)}.$$

The raw attention score between positions $i$ and $j$ is

$$S_{ij} = q_i^\top k_j = x(t_i, p_i)^\top W_K\, x(t_j, p_j),  \quad i,j \in \{1, \ldots, L\}.$$

Causal masking sets $S_{ij} = -\infty$ for $j > i$, enforcing that position $i$ can only attend to earlier tokens (and itself). The attention weights are then

$$P_{ij} = \frac{\exp(S_{ij})}{\displaystyle\sum_{j' \leq i} \exp(S_{ij'})} \quad i,j \in \{1, \ldots, L\}.$$

The contextualised representation at position $i$ is

$$h_i = \sum_{j \leq i} P_{ij}\, v_j, \quad i \in \{1, \ldots, L\}.$$

---

### Score decomposition

Because each $x(t, p)$ has exactly two nonzero coordinates, the score $S_{ij}$ decomposes additively into four interpretable tables:

$$S_{ij} = A_{t_i,\, t_j} + B_{t_i,\, p_j} + C_{p_i,\, t_j} + D_{p_i,\, p_j}, \quad i,j \in \{1, \ldots, L\}$$

where

$$A = E_t^\top W_K E_t \in \mathbb{R}^{V \times V}, \qquad B = E_t^\top W_K E_p \in \mathbb{R}^{V \times L},$$

$$C = E_p^\top W_K E_t \in \mathbb{R}^{L \times V}, \qquad D = E_p^\top W_K E_p \in \mathbb{R}^{L \times L},$$

and $E_t = [e_1 \mid \cdots \mid e_V]$, $E_p = [e_{V+1} \mid \cdots \mid e_{V+L}]$ are the token and position embedding matrices.

---

### Output

Since we require only the output at the final position $i = L$, we need only the last row of $P$:

$$P_{Lj} = \frac{\exp(S_{Lj})}{\displaystyle\sum_{j'=1}^{L} \exp(S_{Lj'})}, \qquad j = 1, \ldots, L.$$

This depends only on the single row

$$S_{Lj} = A_{t_L,\, t_j} + B_{t_L,\, p_j} + C_{L,\, t_j} + D_{L,\, p_j} \qquad j = 1, \ldots, L.$$

The output representation is

$$h_L = \sum_{j=1}^{L} P_{Lj}\, v_j.$$

A learned output projection $W_O \in \mathbb{R}^{V \times d}$ then maps this to logits over the vocabulary:

$$\ell = W_O\, h_L \in \mathbb{R}^V,$$

and the predicted next-token distribution is

$$\hat{y}_k = \frac{\exp(\ell_k)}{\displaystyle\sum_{k'=1}^{V} \exp(\ell_{k'})}, \qquad k \in \{1, \ldots, V\}.$$

---

### Learnable parameters

With $V = 5$ and $L = 4$, the full parameter set is:

| Matrix | Shape | Parameters |
|--------|-------|-----------|
| $A$ | $V \times V$ | $25$ |
| $B$ | $V \times L$ | $20$ |
| $C$ | $L \times V$ | $20$ |
| $D$ | $L \times L$ | $16$ |
| $W_V$ | $d \times d$ | $81$ |
| $W_O$ | $V \times d$ | $45$ |

Since only the last row of the score matrix enters the computation, the effective score parameters reduce from $81$ to $5 + 4 + 5 + 4 = 18$ (minus one for the softmax row-shift symmetry, giving $17$).